In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.stats import ttest_ind, ranksums
from decimal import Decimal
import math
import warnings
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pyarrow.feather as feather
import pyxations as pyx

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 700)
sns.set(font_scale=1.1)
pd.options.mode.chained_assignment = None
warnings.filterwarnings("ignore", category=RuntimeWarning)
from matplotlib.cbook import boxplot_stats

In [ ]:
def pre_processing_from_feather(
    df_samples: pd.DataFrame,
    type_of_saccade: str,
    baseline_start=-200.0,
    baseline_end=100.0,
    interpolate=False,
    threshold_to_accept_sacc=0.5,
    FILTER=1.5,
    filter_by_block=False,
    selected_block=1,
    savgol_flag=False,
) -> dict:
    meta_cols = ['trial_index', 'typeOfSaccade', 'cueShownAtLeft', 'intraEnd', 'isTutorial']

    # One row per trial (all types), sorted by trial_index — mirrors original row order
    all_trials = (
        df_samples[
            (df_samples['isTutorial'] == False) &
            (df_samples['typeOfSaccade'].notna())
        ][meta_cols]
        .drop_duplicates('trial_index')
        .sort_values('trial_index')
        .reset_index(drop=True)
    )

    # Assign block numbers to ALL trials before type filtering (same logic as original)
    blocks = sum([[i] * 20 for i in range(1, 17)], [])
    all_trials['block'] = blocks

    # Filter by type
    df_saccade = all_trials[all_trials['typeOfSaccade'] == type_of_saccade].copy()

    if filter_by_block:
        df_saccade = df_saccade[df_saccade['block'] == selected_block].copy()

    pro_sacc_errors = 0
    anti_sacc_errors = 0
    ts_xs_ys = []
    pro_sacc_errors_rt = []
    pro_sacc_correct_rt = []
    anti_sacc_errors_rt = []
    anti_sacc_correct_rt = []

    for _, trial_row in df_saccade.iterrows():
        t0 = trial_row['intraEnd']
        t_samps = df_samples[
            df_samples['trial_index'] == trial_row['trial_index']
        ].sort_values('tSample')

        xs = t_samps['X'].to_numpy(dtype=float)
        ys = t_samps['Y'].to_numpy(dtype=float)
        ts = t_samps['tSample'].to_numpy(dtype=float) - t0

        # Interpolate (same as original)
        if interpolate:
            fx = interp1d(ts, xs, kind="linear")
            fy = interp1d(ts, ys, kind="linear")
            samples = int((ts[-1] - (-200)) / 30)
            ts_new = np.linspace(-200, ts[-1], samples)
            xs = fx(ts_new)
            ys = fy(ts_new)
            ts = ts_new

        # 1. Baseline median [-200, 100]
        x_base = np.median(xs[(ts > baseline_start) & (ts < baseline_end)])
        y_base = np.median(ys[(ts > baseline_start) & (ts < baseline_end)])

        # 2. Reference window median [500, 700]
        x_max = np.median(xs[(ts > 500.0) & (ts <= 700.0)])
        y_max = np.median(ys[(ts > 500.0) & (ts <= 700.0)])

        # 3. Normalize
        xs = (xs - x_base) / np.abs(x_base - x_max)
        ys = (ys - y_base) / np.abs(y_base - y_max)

        # 4. Mirror if cue was at left
        if trial_row['cueShownAtLeft'] == True:
            xs = xs * -1

        # 5. Reject noisy trials
        if any(xs > FILTER) or any(xs < -FILTER):
            continue

        # 6. Savitzky-Golay filter (optional)
        if savgol_flag:
            xs = savgol_filter(xs, 5, 2)

        # 7. Detect direction by threshold crossing
        xs_after_baseline = xs[ts > baseline_end]
        ts_after_baseline = ts[ts > baseline_end]

        if type_of_saccade == "prosaccade":
            is_sacc_error_in_trial = np.any(xs_after_baseline < -threshold_to_accept_sacc)
            if is_sacc_error_in_trial:
                pro_sacc_errors += 1
                err_idx = np.where(xs_after_baseline < -threshold_to_accept_sacc)[0][0]
                pro_sacc_errors_rt.append(f"{ts_after_baseline[err_idx]:.2f}")
            else:
                corr_idx = np.where(xs_after_baseline >= threshold_to_accept_sacc)[0]
                if len(corr_idx):
                    pro_sacc_correct_rt.append(f"{ts_after_baseline[corr_idx[0]]:.2f}")

        elif type_of_saccade == "antisaccade":
            is_anti_sacc_error_in_trial = np.any(xs_after_baseline > threshold_to_accept_sacc)
            if is_anti_sacc_error_in_trial:
                anti_sacc_errors += 1
                err_idx = np.where(xs_after_baseline > threshold_to_accept_sacc)[0][0]
                anti_sacc_errors_rt.append(f"{ts_after_baseline[err_idx]:.2f}")
            else:
                corr_idx = np.where(xs_after_baseline <= -threshold_to_accept_sacc)[0]
                if len(corr_idx):
                    anti_sacc_correct_rt.append(f"{ts_after_baseline[corr_idx[0]]:.2f}")

        ts_xs_ys.append((ts, xs, ys))

    try:
        trials_rejected = 100 - (len(ts_xs_ys) / len(df_saccade) * 100)
        pro_saccades_errors_perc = (pro_sacc_errors / len(ts_xs_ys)) * 100
        anti_saccades_errors_perc = (anti_sacc_errors / len(ts_xs_ys)) * 100
    except ZeroDivisionError:
        trials_rejected = 100
        pro_saccades_errors_perc = np.nan
        anti_saccades_errors_perc = np.nan

    return {
        "ts_xs_ys": ts_xs_ys,
        "number_of_trials_remained": len(ts_xs_ys),
        "pro_sacc_errors": pro_sacc_errors,
        "pro_sacc_errors_perc": pro_saccades_errors_perc,
        "anti_sacc_errors": anti_sacc_errors,
        "anti_sacc_errors_perc": anti_saccades_errors_perc,
        "trials_rejected": trials_rejected,
        "pro_sacc_errors_rt": pro_sacc_errors_rt,
        "pro_sacc_correct_rt": pro_sacc_correct_rt,
        "anti_sacc_errors_rt": anti_sacc_errors_rt,
        "anti_sacc_correct_rt": anti_sacc_correct_rt,
    }


def one_subject(df, suj_number, type_of_saccade):
    fig, axs = plt.subplots(
        3, 1, height_ratios=[1, 4, 1], sharex=True, constrained_layout=True
    )
    suj_number = str(suj_number)
    df = df.query("subject == @suj_number")
    for i in df.query("subject == @suj_number")[type_of_saccade].iloc[0]:
        ts, xs, ys = i[0], i[1], i[2]
        axs[1].plot(ts, xs)
        axs[1].axhline(y=0.5, color="k", linestyle="-")
        axs[1].axhline(y=-0.5, color="k", linestyle="-")
        axs[1].axvline(x=0, color="k", linestyle="-")
        axs[1].set_ylabel("x coordinate predictions")
        axs[1].set_xticks(np.arange(-200, 1000, step=100))
        axs[1].set_xlim(-200, 700)
        axs[1].set_ylim(-2, 2)
    if type_of_saccade == "prosaccade":
        data_error = [float(i) for i in df["pro_sacc_errors_rt"].iloc[0]]
        data_correct = [float(i) for i in df["pro_sacc_correct_rt"].iloc[0]]
        sns.kdeplot(data_error, ax=axs[2])
        sns.rugplot(data_error, ax=axs[2], height=0.1)
        sns.kdeplot(data_correct, ax=axs[0])
        sns.rugplot(data_correct, ax=axs[0], height=0.1)
        axs[2].set_xlim([-200, 700])
        axs[2].invert_yaxis()
        axs[0].set_xlim([-200, 700])
    else:
        data_error = [float(i) for i in df["anti_sacc_errors_rt"].iloc[0]]
        data_correct = [float(i) for i in df["anti_sacc_correct_rt"].iloc[0]]
        sns.kdeplot(data_error, ax=axs[0])
        sns.rugplot(data_error, ax=axs[0], height=0.1)
        sns.kdeplot(data_correct, ax=axs[2])
        sns.rugplot(data_correct, ax=axs[2], height=0.1)
        axs[0].set_xlim([-200, 700])
        axs[2].set_xlim([-200, 700])
        axs[2].invert_yaxis()
    plt.suptitle(f"{type_of_saccade}")
    fig.align_ylabels(axs[:])
    fig.supxlabel("time (ms)")
    plt.show()

def calculate_distance(x_positions):
    distances = np.diff(x_positions)
    distance = np.sum(np.sqrt(distances ** 2))
    return distance

In [ ]:
RAW_DATA   = Path('./raw_data')
OUTPUT_DIR = Path('.')
BIDS_NAME  = 'antisaccades_bids'
SCREEN_W, SCREEN_H = 1366, 768

BEHAVIORAL_COLS = [
    'typeOfSaccade', 'cueShownAtLeft',
    'itiEnd', 'fixEnd', 'intraEnd', 'visualEnd', 'responseEnd',
    'viewportWidth', 'isTutorial',
]

bids_path = pyx.dataset_to_bids(
    OUTPUT_DIR, RAW_DATA, BIDS_NAME, format_name='webgazer'
)

pyx.compute_derivatives_for_dataset(
    bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    overwrite=True,
    screen_height=SCREEN_H,
    screen_width=SCREEN_W,
    behavioral_columns=BEHAVIORAL_COLS,
)

derivatives_path = bids_path.with_name(bids_path.name + '_derivatives')
print(f'Derivatives: {derivatives_path}')

### Load all files and **separate in blocks**

In [ ]:
all_dfs = []

print("processing ...")
for ses_dir in sorted((derivatives_path / 'sub-0001').iterdir()):
    samples_file = ses_dir / 'samples.feather'
    if not samples_file.exists():
        continue

    suj = ses_dir.name[4:]  # "ses-100" -> "100"
    df_samples = feather.read_feather(samples_file)

    subjects = []
    prosaccade_trials_remained = []
    antisaccade_trials_remained = []
    pro_saccades = []
    anti_saccades = []
    pro_saccades_errors = []
    pro_saccades_errors_perc = []
    anti_saccades_errors = []
    anti_saccades_errors_perc = []
    trials_rejected_prosaccade = []
    trials_rejected_antisaccade = []
    pro_sacc_errors_rt = []
    pro_sacc_correct_rt = []
    anti_sacc_errors_rt = []
    anti_sacc_correct_rt = []
    block_number = []

    print(suj)

    for block in range(1, 17):
        anti_saccade_dict = pre_processing_from_feather(
            df_samples,
            type_of_saccade="antisaccade",
            interpolate=True,
            filter_by_block=True,
            selected_block=block,
        )
        pro_saccade_dict = pre_processing_from_feather(
            df_samples,
            type_of_saccade="prosaccade",
            interpolate=True,
            filter_by_block=True,
            selected_block=block,
        )
        subjects.append(suj)
        pro_saccades.append(pro_saccade_dict["ts_xs_ys"])
        anti_saccades.append(anti_saccade_dict["ts_xs_ys"])
        prosaccade_trials_remained.append(pro_saccade_dict["number_of_trials_remained"])
        antisaccade_trials_remained.append(anti_saccade_dict["number_of_trials_remained"])
        pro_saccades_errors.append(pro_saccade_dict["pro_sacc_errors"])
        pro_saccades_errors_perc.append(pro_saccade_dict["pro_sacc_errors_perc"])
        anti_saccades_errors.append(anti_saccade_dict["anti_sacc_errors"])
        anti_saccades_errors_perc.append(anti_saccade_dict["anti_sacc_errors_perc"])
        trials_rejected_prosaccade.append(pro_saccade_dict["trials_rejected"])
        trials_rejected_antisaccade.append(anti_saccade_dict["trials_rejected"])
        pro_sacc_errors_rt.append(pro_saccade_dict["pro_sacc_errors_rt"])
        pro_sacc_correct_rt.append(pro_saccade_dict["pro_sacc_correct_rt"])
        anti_sacc_errors_rt.append(anti_saccade_dict["anti_sacc_errors_rt"])
        anti_sacc_correct_rt.append(anti_saccade_dict["anti_sacc_correct_rt"])
        block_number.append(block)

    _df = pd.DataFrame({
        "subject":                              subjects,
        "prosaccade":                           pro_saccades,
        "antisaccade":                          anti_saccades,
        "prosaccade_errors":                    pro_saccades_errors,
        "antisaccade_errors":                   anti_saccades_errors,
        "delta_errors":                         np.array(anti_saccades_errors) - np.array(pro_saccades_errors),
        "trials_rejected_antisaccade_percentage": trials_rejected_antisaccade,
        "trials_rejected_prosaccade_percentage":  trials_rejected_prosaccade,
        "pro_sacc_errors_rt":                   pro_sacc_errors_rt,
        "pro_sacc_correct_rt":                  pro_sacc_correct_rt,
        "anti_sacc_errors_rt":                  anti_sacc_errors_rt,
        "anti_sacc_correct_rt":                 anti_sacc_correct_rt,
        "prosaccade_errors_perc":               pro_saccades_errors_perc,
        "antisaccade_errors_perc":              anti_saccades_errors_perc,
        "prosaccade_trials_remained":           prosaccade_trials_remained,
        "antisaccade_trials_remained":          antisaccade_trials_remained,
        "block":                                block_number,
    })

    _df["pro_sacc_errors_rt_median"] = _df["pro_sacc_errors_rt"].apply(
        lambda x: np.median([float(i) for i in x]) if x else np.nan
    )
    _df["pro_sacc_correct_rt_median"] = _df["pro_sacc_correct_rt"].apply(
        lambda x: np.median([float(i) for i in x]) if x else np.nan
    )
    _df["anti_sacc_errors_rt_median"] = _df["anti_sacc_errors_rt"].apply(
        lambda x: np.median([float(i) for i in x]) if x else np.nan
    )
    _df["anti_sacc_correct_rt_median"] = _df["anti_sacc_correct_rt"].apply(
        lambda x: np.median([float(i) for i in x]) if x else np.nan
    )

    all_dfs.append(_df)

df_all_blocks = pd.concat(all_dfs)
print("\ndf_all_blocks ready ✅")

### Filtering blocks

In [ ]:
# Constants
MAX_NUMBER_INCORRECTS_BY_BLOCK = (
    10  # Esto implica que en un bloque hizo al menos el 50% mal
)
FIRST_BLOCK = 2
LAST_BLOCK = 8

# Filter
df_blocks_filtered = (
    df_all_blocks.query(
        "prosaccade_errors < @MAX_NUMBER_INCORRECTS_BY_BLOCK and antisaccade_errors < @MAX_NUMBER_INCORRECTS_BY_BLOCK and @FIRST_BLOCK <= block <= @LAST_BLOCK"
    )  # Con la mitad de los bloques
    .groupby("subject")
    .agg(
        {
            "prosaccade": list,
            "antisaccade": list,
            "prosaccade_errors": "sum",
            "antisaccade_errors": "sum",
            "prosaccade_trials_remained": "sum",
            "antisaccade_trials_remained": "sum",
            "pro_sacc_errors_rt": "sum",
            "pro_sacc_correct_rt": "sum",
            "anti_sacc_errors_rt": "sum",
            "anti_sacc_correct_rt": "sum",
            "pro_sacc_errors_rt_median": np.nanmean,
            "anti_sacc_errors_rt_median": np.nanmean,
            "pro_sacc_correct_rt_median": np.nanmean,
            "anti_sacc_correct_rt_median": np.nanmean,
            "block": lambda x: x.nunique(),
        }
    )
)

df_blocks_filtered = df_blocks_filtered.rename({"block": "remained_blocks"}, axis=1)
df_blocks_filtered["max_number_incorrect_by_block"] = MAX_NUMBER_INCORRECTS_BY_BLOCK
df_blocks_filtered.insert(
    2,
    "delta_errors",
    df_blocks_filtered["antisaccade_errors"] - df_blocks_filtered["prosaccade_errors"],
)


# Percentage
df_blocks_filtered["prosaccade_errors_perc"] = (
    df_blocks_filtered["prosaccade_errors"]
    / df_blocks_filtered["prosaccade_trials_remained"]
) * 100


df_blocks_filtered["antisaccade_errors_perc"] = (
    df_blocks_filtered["antisaccade_errors"]
    / df_blocks_filtered["antisaccade_trials_remained"]
) * 100

print("df_blocks_filtered ready ✅")

### Plot stuff

In [ ]:
# Plot errors
ax = sns.boxplot(
    data=df_blocks_filtered[["prosaccade_errors_perc", "antisaccade_errors_perc"]],
    width=0.3,
)
ax.set_ylabel("% of incorrect trials")
labels = [item.get_text() for item in ax.get_xticklabels()]
labels = ['prosaccade', 'antisaccade']
ax.set_xticklabels(labels)

boxes = ax.patches

for i, box in enumerate(boxes):
    box.set_facecolor("w")


p_value = round((
    ranksums(
        df_blocks_filtered["prosaccade_errors_perc"],
        df_blocks_filtered["antisaccade_errors_perc"],
    )[1]
), 4)

print(p_value)
print(f"N={len(df_blocks_filtered)}, blocks={df_blocks_filtered['remained_blocks'].max()}")

print(f"Cantidad de bloques filtrados: {(len(df_blocks_filtered['remained_blocks']) * 16)  - df_blocks_filtered['remained_blocks'].sum()}")

df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]]

In [ ]:
print("means:") 
print(df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]].loc['mean',:].round(1), end='\n\n')
print("std:") 
print(df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]].loc['std',:].round(1), end='\n\n')
print("median:") 
print(df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]].loc['50%',:].round(1), end='\n\n')
print("25%:",) 
print(df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]].loc['25%',:].round(1), end='\n\n')
print("75%:") 
print(df_blocks_filtered.describe()[["prosaccade_errors_perc", "antisaccade_errors_perc"]].loc['75%',:].round(1), end='\n\n')

In [ ]:
# Incorrectos por bloque
all_dfs = []
for last_block in range(1, 17):
    FIRST_BLOCK = 1
    LAST_BLOCK = last_block

    _df_blocks_filtered = (
        df_all_blocks.query(
            "prosaccade_errors < @MAX_NUMBER_INCORRECTS_BY_BLOCK and antisaccade_errors < @MAX_NUMBER_INCORRECTS_BY_BLOCK and @FIRST_BLOCK <= block <= @LAST_BLOCK"
        )
        .groupby("subject")
        .agg(
            {
                "prosaccade_errors": "sum",
                "antisaccade_errors": "sum",
                "prosaccade_trials_remained": "sum",
                "antisaccade_trials_remained": "sum",
                "pro_sacc_errors_rt": "sum",
                "pro_sacc_correct_rt": "sum",
                "anti_sacc_errors_rt": "sum",
                "anti_sacc_correct_rt": "sum",
                "pro_sacc_errors_rt_median": np.nanmean,
                "anti_sacc_errors_rt_median": np.nanmean,
                "pro_sacc_correct_rt_median": np.nanmean,
                "anti_sacc_correct_rt_median": np.nanmean,
                "block": lambda x: x.nunique(),
            }
        )
    )

    _df_blocks_filtered = _df_blocks_filtered.rename({"block": "remained_blocks"}, axis=1)

    _df_blocks_filtered["prosaccade_errors_perc"] = (
        _df_blocks_filtered["prosaccade_errors"]
        / _df_blocks_filtered["prosaccade_trials_remained"]
    ) * 100

    _df_blocks_filtered["antisaccade_errors_perc"] = (
        _df_blocks_filtered["antisaccade_errors"]
        / _df_blocks_filtered["antisaccade_trials_remained"]
    ) * 100
    all_dfs.append(_df_blocks_filtered)

all_dataframes = pd.concat(all_dfs)

ax = sns.boxplot(
    data=all_dataframes.melt(id_vars=['remained_blocks'], value_vars=['prosaccade_errors_perc', 'antisaccade_errors_perc'],
                 var_name='Type of trial', value_name='% of incorrect trials'),
    x='remained_blocks',
    y='% of incorrect trials',
    hue='Type of trial',
    showfliers=True
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

ax = sns.boxplot(
    data=all_dataframes.melt(id_vars=['remained_blocks'], value_vars=['prosaccade_errors_perc', 'antisaccade_errors_perc'],
                 var_name='Type of trial', value_name='% of incorrect trials'),
    x='remained_blocks',
    y='% of incorrect trials',
    hue='Type of trial',
    showfliers=False
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# RT VERSION CON BLOQUES (Plot original)
ax = sns.boxplot(
    data=df_blocks_filtered[
        [
            "pro_sacc_errors_rt_median",
            "anti_sacc_errors_rt_median",
            "pro_sacc_correct_rt_median",
            "anti_sacc_correct_rt_median",
        ]
    ],
)

boxes = ax.patches

for i, box in enumerate(boxes):
    box.set_facecolor("w")

plt.ylabel("time (ms)")
ax.set_xticklabels(
    [
        "Prosaccade\nincorrect",
        "Antisaccade\nincorrect",
        "Prosaccade\ncorrect",
        "Antisaccade\ncorrect",
    ]
)

plt.setp(ax.artists, edgecolor="k", facecolor="w")
plt.setp(ax.lines, color="k")
plt.show()
df_blocks_filtered.describe()

In [ ]:
# RT VERSION CON BLOQUES
data = {
    'Direction': np.concatenate([
        ['Prosaccade']*len(df_blocks_filtered),
        ['Antisaccade']*len(df_blocks_filtered),
        ['Prosaccade']*len(df_blocks_filtered),
        ['Antisaccade']*len(df_blocks_filtered),
    ]),
    'Response': np.concatenate([
        ['Incorrect']*len(df_blocks_filtered),
        ['Incorrect']*len(df_blocks_filtered),
        ['Correct']*len(df_blocks_filtered),
        ['Correct']*len(df_blocks_filtered),
    ]),
    'RT': np.concatenate([
        df_blocks_filtered["pro_sacc_errors_rt_median"],
        df_blocks_filtered["anti_sacc_errors_rt_median"],
        df_blocks_filtered["pro_sacc_correct_rt_median"],
        df_blocks_filtered["anti_sacc_correct_rt_median"],
    ])
}
df_plot = pd.DataFrame(data)
df_plot['Condition'] = df_plot['Direction'] + '_' + df_plot['Response']

plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

palette = {'Prosaccade_Incorrect': (30/255, 144/255, 255/255, 0.2),
           'Antisaccade_Incorrect': (240/255, 128/255, 128/255, 0.2),
           'Prosaccade_Correct': (30/255, 144/255, 255/255, 0.2),
           'Antisaccade_Correct': (240/255, 128/255, 128/255, 0.2)}

ax = sns.boxplot(x='Condition', y='RT', hue='Condition', data=df_plot,
    width=0.25,
    fliersize=0,
    palette=palette,
)

ax = sns.swarmplot(x='Condition', y='RT', hue='Condition', data=df_plot,
    color='black',
    size=5,
    legend=False,
)

plt.xlabel("")
plt.xticks([])
plt.ylabel("mean RT (s)", fontsize=12, font='Ubuntu')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

In [ ]:
rt_lm = ols('RT ~ Direction*Response',data=df_plot).fit()
table = sm.stats.anova_lm(rt_lm, typ=2) # Type 2 ANOVA DataFrame
print(table)

In [ ]:
df_blocks_filtered[
        [
            "pro_sacc_errors_rt_median",
            "anti_sacc_errors_rt_median",
            "pro_sacc_correct_rt_median",
            "anti_sacc_correct_rt_median",
        ]
    ].describe()